# Notebook 10 — B1-M2: Per-target ablation matrix (5-symbol × 8-year slice)

**Project B1 — target reframing.** Phase 4A proved next-bar/1-day *signed return*
is structurally unlearnable from the current feature set on the Dow-30+ETF sandbox
(`docs/PHASE_4A_REPORT.md`). B1 isolates the **target** axis: is a *different
prediction object* — drawdown risk, realized vol, or a longer directional horizon —
more learnable from the same information set?

This notebook runs the **provisional slice ablation** (METHODOLOGY §11 — slice
verdict is provisional; B1-M3 is the confirmatory full-panel run). For each of the
four pre-committed targets it compares a **GBM** against that target's
**pre-committed baseline**, per required regime, and renders the verdict through
`b1_gate_report` (the source of truth shipped in B1-M1, METHODOLOGY §2).

### The four targets (`features/targets.py`, thresholds pinned in code, METHODOLOGY §1)

| # | Target | Type | Metric | Baseline | Deflation |
|---|---|---|---|---|---|
| T1 | `drawdown_21d` — P(max DD > 5% over next 21 bars) | classification | ROC-AUC | climatology base-rate + ARIMA-vol-implied DD probability | skill-z |
| T2 | `realized_vol_21d` — log realized vol next 21 bars | regression | MAE | EWMA(0.94) + ARIMA-on-log-vol | skill-z |
| T3 | `directional_5d` — sign(ret_5d) | classification | AUC **and** tradeable Sharpe | majority-class + ARIMA sign | DSR |
| T4 | `directional_21d` — sign(ret_21d) | classification | AUC **and** tradeable Sharpe | majority-class + ARIMA sign | DSR |

### The gate (`b1_gate_report`, conjunction of three stages)
1. **Materiality** — every criterion in `spec.materiality` met in >= `min_pass`(=2) of
   the required regimes `(qe_bull, covid, rate_cycle)`.
2. **Significance** — paired stationary-block-bootstrap (21-day blocks) 90% CI of the
   gated-metric delta **excludes 0** in >= 1 required regime.
3. **Deflation** — deflated Sharpe > 0 (directional) or the skill-z analog > 0 (T1/T2),
   with the deflation N read from the trial-count ledger.

> **Slice caveat (declared up front).** With `train_window=504` (~2y) on a 2018-start
> slice, the OOS window begins ~2020, so **`qe_bull` (<=2019) has zero OOS bars on this
> slice** — exactly as every Phase 4A slice notebook. The slice therefore speaks only to
> `covid` and `rate_cycle`; `min_pass=2` then requires *both* to pass. B1-M3's full
> panel (2004-2026) restores all three regimes. The slice verdict is **provisional**.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, roc_auc_score

import quant.storage.catalog as catalog
from quant.backtest.harness import run_portfolio_backtest
from quant.backtest.metrics import compute_metrics
from quant.backtest.regime_metrics import DEFAULT_REGIMES_REQUIRED, b1_gate_report
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.statistics import (
    bootstrap_metric_delta_ci,
    bootstrap_sharpe_delta_ci,
    deflated_sharpe_ratio,
    forecast_skill_z,
)
from quant.backtest.target_eval import (
    collect_oos_predictions,
    simulate_signal_returns,
)
from quant.features.cross_sectional import add_cross_sectional_features
from quant.features.engineering import build_features
from quant.features.targets import (
    DRAWDOWN_HORIZON,
    DRAWDOWN_THRESHOLD,
    TARGET_CATALOG,
    make_target_labels,
)
from quant.ledger import cumulative_trial_count
from quant.models.arima_baseline import ARIMABaseline
from quant.models.gbm import GBMModel

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# ── Pinned config (METHODOLOGY §1 — fixed before any compute) ────────────────
DEMO_SYMBOLS = ["AAPL", "MSFT", "JPM", "JNJ", "SPY"]
DEMO_START, DEMO_END = "2018-01-02", "2026-04-21"
TRAIN_W, TEST_W, STEP, EMBARGO = 504, 63, 63, 3
GBM_PREVIEW = dict(n_iter=10, n_splits=3, random_state=7)      # slice preview
REGIMES = DEFAULT_REGIMES_REQUIRED                             # (qe_bull, covid, rate_cycle)
MIN_PASS = 2
BOOT = dict(block_len=21, n_boot=1000, ci=0.90, seed=0)        # T1 21-day-block convention
EWMA_LAMBDA = 0.94                                            # RiskMetrics persistence
# Self-comparison count this matrix adds (4 targets x 3 required regimes) — pinned
# before results so the DSR-sensitivity note below is not chosen post hoc.
N_SELF_COMPARISONS = len(TARGET_CATALOG) * len(REGIMES)

print("targets:", list(TARGET_CATALOG))
print("required regimes:", REGIMES, " min_pass:", MIN_PASS)
print("walk-forward:", dict(train=TRAIN_W, test=TEST_W, step=STEP, embargo=EMBARGO))


## 1 · Load the slice OHLCV from the lake

Same recipe as the Phase 4A slice notebooks (`equity_eod_tiingo`, tz stripped to
naive). If the lake is unavailable the notebook degrades to a clearly-flagged
synthetic panel so it still executes end-to-end — but the **verdict is only valid
on real data** (`REAL_DATA == True`).

In [ ]:
REAL_DATA = False
prices_by_sym: dict[str, pd.DataFrame] = {}
try:
    syms_sql = ", ".join(f"'{s}'" for s in DEMO_SYMBOLS)
    eq = catalog.query(f'''
        SELECT symbol, timestamp, open, high, low, close, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
          AND timestamp BETWEEN '{DEMO_START}' AND '{DEMO_END}'
        ORDER BY symbol, timestamp
    ''')
    eq["timestamp"] = pd.to_datetime(eq["timestamp"]).dt.tz_localize(None)
    eq = eq.set_index("timestamp")
    for s in DEMO_SYMBOLS:
        sdf = eq[eq["symbol"] == s][["open", "high", "low", "close", "volume"]].copy()
        sdf = sdf[~sdf.index.duplicated(keep="last")].sort_index()
        if not sdf.empty:
            prices_by_sym[s] = sdf
    REAL_DATA = len(prices_by_sym) == len(DEMO_SYMBOLS)
    print(f"Loaded OHLCV for {len(prices_by_sym)}/{len(DEMO_SYMBOLS)} symbols (REAL_DATA={REAL_DATA})")
    for s, df in prices_by_sym.items():
        print(f"  {s}: {len(df):,} bars  {df.index.min().date()} -> {df.index.max().date()}")
except Exception as exc:  # noqa: BLE001
    print(f"Lake unavailable ({type(exc).__name__}): {exc}")

if not REAL_DATA:
    print("\n[FALLBACK] Building a synthetic panel so the notebook executes; "
          "VERDICT IS NOT VALID in this mode.")
    dates = pd.bdate_range(DEMO_START, periods=2085)
    prices_by_sym = {}
    for i, s in enumerate(DEMO_SYMBOLS):
        r = np.random.default_rng(i)
        close = 100.0 * np.exp(np.cumsum(r.normal(0.0003, 0.012, len(dates))))
        open_ = close * (1 + r.uniform(-0.002, 0.002, len(dates)))
        prices_by_sym[s] = pd.DataFrame(
            {"open": open_, "high": np.maximum(close, open_) * 1.003,
             "low": np.minimum(close, open_) * 0.997, "close": close,
             "volume": r.integers(500_000, 2_000_000, len(dates)).astype(float)},
            index=dates,
        )


## 2 · Build the M6 feature set (the held-fixed input)

B1 holds the **feature set fixed** and varies only the **target**. The slice uses
the same 24-column no-sentiment M6 set as every Phase 4A slice notebook
(`build_features` with corrected FRED publication-lag joins + cross-sectional
ranks; `sentiment_df=None` on the slice). **Declared deviation (METHODOLOGY §9):**
the PRD names the "25-column M6 set" — the 3 sentiment columns are excluded on the
slice exactly as in Phase 4A's slice work; B1-M3's full panel restores them.

In [ ]:
feats_raw = build_features(DEMO_SYMBOLS, prices_by_sym)  # corrected FRED joins (M5 default)
for s in DEMO_SYMBOLS:
    # build_features returns UTC tz-aware indexes (FRED ASOF join); strip to naive.
    feats_raw[s].index = feats_raw[s].index.tz_localize(None)

feats_xs = add_cross_sectional_features(feats_raw, min_symbols=5)
FEATURE_COLS = list(feats_xs[DEMO_SYMBOLS[0]].columns)
print(f"feature columns ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)


## 3 · Invariant-4 check on the real slice config (B1-M2 note (c))

`backtest/CLAUDE.md` invariant 4: *test-fold length must be much greater than
`label_horizon + embargo`*, else purge would decimate training. The new 5- and
21-bar horizons exceed Phase 4A's 1-bar label, so we **assert** the hard bound and
**flag** (never silently shrink) the tightest case. We also report the fraction of
each test fold that carries a *complete* forward label window (the last `horizon`
bars of every fold are NaN-labelled and drop out of the OOS panel).

In [ ]:
rows = []
hard_ok = True
for tid, spec in TARGET_CATALOG.items():
    h = spec.horizon_bars
    need = h + EMBARGO
    ratio = TEST_W / need
    label_complete_frac = max(0.0, (TEST_W - h)) / TEST_W
    hard = TEST_W > need
    hard_ok = hard_ok and hard
    rows.append({
        "target": tid, "horizon": h, "label_horizon+embargo": need,
        "test_window": TEST_W, "ratio": round(ratio, 2),
        "hard_ok": hard, "valid_label_frac": round(label_complete_frac, 3),
        "much_greater_2x": ratio >= 2.0,
    })
inv4 = pd.DataFrame(rows).set_index("target")
display(inv4)
assert hard_ok, "Invariant 4 HARD bound violated: test_window <= label_horizon+embargo"
tight = inv4[~inv4["much_greater_2x"]].index.tolist()
if tight:
    print(f"[FLAG] horizons {tight} clear the hard bound but are < 2x. Not shrinking the "
          "training set; ~1/3 of each fold's labels are NaN-dropped (valid_label_frac "
          "above). B1-M3's full panel (longer test folds) relaxes this.")


## 4 · Build aligned panels per target

For each target we build NaN-free `features / labels / prices` dicts with the
single-`dropna` + intersection discipline (so every arm sees identical rows). The
directional targets also carry a forward-**return** label series for the ARIMA
tradeable-Sharpe baseline (the Phase-4A ARIMA control).

In [ ]:
def make_panel(label_fn):
    """Return aligned (features, labels, prices) over the single-dropna common index."""
    feats, labels, prices = {}, {}, {}
    for s in DEMO_SYMBOLS:
        X = feats_xs[s].dropna()
        y = label_fn(s).dropna()
        common = X.index.intersection(y.index)
        feats[s] = X.loc[common]
        labels[s] = y.loc[common]
        prices[s] = prices_by_sym[s].loc[common]
    return feats, labels, prices


def fwd_return(close, horizon):
    return close.shift(-horizon) / close - 1.0


def target_label_fn(tid):
    return lambda s: make_target_labels(tid, prices_by_sym[s]["close"]).series


# ── metric_fns ──
def _auc(y, p):
    y = np.asarray(y)
    if len(np.unique(y[~np.isnan(y)])) < 2:
        raise ValueError("single class — AUC undefined")
    return float(roc_auc_score(y, p))

def _mae(y, p):
    return float(mean_absolute_error(y, p))

def _auc_cs(arr):   # column_stack([y_true, score]) -> AUC, for the bootstrap
    return _auc(arr[:, 0], arr[:, 1])

def _mae_cs(arr):
    return _mae(arr[:, 0], arr[:, 1])


def _auc_safe(y, score):
    """ROC-AUC dropping NaN scores; (nan, mask) when a regime pool is degenerate."""
    ok = ~np.isnan(score)
    if not ok.any() or len(np.unique(y[ok])) < 2:
        return np.nan, ok
    try:
        return _auc(y[ok], score[ok]), ok
    except ValueError:
        return np.nan, ok


def regime_mask(frame, regime):
    # Tag from the frame's OWN dates: each target's OOS calendar differs slightly
    # (different horizon -> different label-NaN trimming), so a shared label Series
    # would silently drop rows on dates absent from it.
    lab = tag_regimes(pd.DatetimeIndex(frame.index), DateRangeDetector())
    return (lab.to_numpy() == regime)


def ewma_logvol(close, lam=EWMA_LAMBDA):
    """Causal EWMA daily-vol persistence forecast, as log-vol (T2/T1 baseline)."""
    r = close.pct_change()
    var = r.pow(2).ewm(alpha=1 - lam, adjust=False).mean()
    vol = np.sqrt(var)
    return np.log(vol.replace(0.0, np.nan))


def dd_prob_from_logvol(logvol, dd_threshold=DRAWDOWN_THRESHOLD, horizon=DRAWDOWN_HORIZON):
    """Vol-implied ``P(max drawdown > dd_threshold over `horizon` bars)`` — parametric.

    The principled T1 baseline (B1 PRD T1 row): a per-bar **log-vol forecast** mapped
    to a genuine drawdown probability via the driftless-Gaussian random-walk
    running-minimum (reflection-principle) approximation
    ``P(min cum-return < -d) = 2 * Phi(-d / (sigma * sqrt(h)))`` with
    ``sigma = exp(logvol)``. It is **monotone increasing in sigma**, so the ROC-AUC of
    this probability equals the AUC of the raw vol ranking (AUC is rank-based) — but
    the score is now a calibrated probability in [0, 1] rather than a raw log-vol
    number, which is what the PRD's "vol-implied DD probability" names (the M2 slice
    used the raw EWMA log-vol score directly: PRIORITIES ``B1-DD-VOLIMPLIED-BASELINE``).

    The map uses the cumulative-return running minimum (min-from-entry), a first-order
    proxy for the peak-to-trough drawdown the T1 label actually measures; it
    *under-states* true drawdown probability, a monotone gap the AUC gate is invariant
    to (declared deviation, METHODOLOGY §9). NaN forecasts pass through as NaN.
    """
    sigma = np.exp(np.asarray(logvol, dtype=float))
    horizon_vol = sigma * np.sqrt(horizon)
    with np.errstate(divide="ignore", invalid="ignore"):
        prob = 2.0 * norm.cdf(-dd_threshold / horizon_vol)
    return np.clip(prob, 0.0, 1.0)


def align_other(base, other):
    """y_pred from another collected frame, aligned onto base's (date,symbol) rows."""
    o = other.assign(_d=other.index).set_index(["_d", "symbol"])["y_pred"]
    keys = pd.MultiIndex.from_arrays([base.index, base["symbol"].to_numpy()])
    return o.reindex(keys).to_numpy()


def align_series(base, series_by_sym):
    """y_pred from a per-symbol Series, aligned onto base's (date,symbol) rows."""
    out = np.full(len(base), np.nan)
    for i, (dt, s) in enumerate(zip(base.index, base["symbol"].to_numpy())):
        ser = series_by_sym.get(s)
        if ser is not None and dt in ser.index:
            out[i] = ser.loc[dt]
    return out


### Regime composition of the slice OOS panel

We tag the OOS dates produced by a single GBM run (all targets share the
walk-forward config, so the OOS calendar is the same up to per-target label-NaN
trimming). This confirms which required regimes actually carry slice data.

In [ ]:
feats5, lab5, px5 = make_panel(target_label_fn("directional_5d"))
print("Fitting GBM on T3 (directional_5d) to establish the OOS calendar ...")
gbm5 = collect_oos_predictions(
    GBMModel(label_horizon=TARGET_CATALOG["directional_5d"].horizon_bars, **GBM_PREVIEW),
    feats5, lab5, train_window=TRAIN_W, test_window=TEST_W, step=STEP,
    label_horizon=TARGET_CATALOG["directional_5d"].horizon_bars, embargo=EMBARGO,
)
oos_regimes = tag_regimes(pd.DatetimeIndex(gbm5.index.unique()), DateRangeDetector())
present = tag_regimes(pd.DatetimeIndex(gbm5.index), DateRangeDetector())
print("\nOOS regime composition (bar-rows across symbols):")
print(present.value_counts().rename("rows").to_frame())
populated = [r for r in REGIMES if (present == r).any()]
print(f"\nRequired regimes populated on this slice: {populated}")
print(f"(empty required regimes — slice cannot speak to them: "
      f"{[r for r in REGIMES if r not in populated]})")


## 5 · Per-target scoring

Each target produces `per_regime_metrics`, the gated-metric `significance_ci`, and
`deflation_passed`, then `b1_gate_report` renders the verdict. The deflation N is
read from the ledger; B1-M2 does **not** append a ledger entry (provisional slice —
B1-M3's runner writes the authoritative entries per A-LEDGER-RUNNERS). For the
directional DSR we deflate against `N + N_SELF_COMPARISONS` so the slice already
penalizes this matrix's search width (pinned above before any result).

In [ ]:
LEDGER_N = cumulative_trial_count()
LEDGER_N_SELF = LEDGER_N + N_SELF_COMPARISONS
print(f"ledger N (current cumulative trials): {LEDGER_N}")
print(f"deflation N used (current + self-comparisons): {LEDGER_N_SELF}")
gate_results = {}
score_detail = {}


### T1 baseline — formalized to the PRD's "ARIMA-vol-implied DD probability" (`B1-DD-VOLIMPLIED-BASELINE`)

The B1 PRD pins the T1 baseline as **climatology base-rate + ARIMA-vol-implied DD
probability**. The original M2 slice shorthanded the second term as a raw
EWMA(0.94)-log-vol score ranked against the 0/1 label (a defensible monotone proxy,
but neither an ARIMA-on-vol forecast nor a calibrated probability — declared as a
§9 deviation in `B1_REPORT.md`). This cell formalizes it:

- **ARIMA-vol-implied DD probability** — an `ARIMA(1,0,0)`-on-log-vol OOS forecast
  (collected on the T1 panel with the same purged walk-forward) mapped through
  `dd_prob_from_logvol`, the parametric driftless-Gaussian drawdown-probability
  function. This is the PRD's named baseline, now a genuine probability in [0, 1].
- The **EWMA-vol proxy** (mapped through the same `dd_prob_from_logvol`, so its AUC is
  identical to the M2 run) is **retained as a floor**: the gate baseline is the
  **best-of {climatology, ARIMA-vol-implied DD probability, EWMA-vol DD probability}**
  per regime. Best-of guarantees the formalization can only *raise* the baseline bar
  versus M2, never lower it — so it can only **strengthen** the clean T1 negative,
  never manufacture a GBM edge (METHODOLOGY §1/§9).

Scope is the **slice notebook only** (this task's `deliverable`); the full-panel
runner `run_b1_arms.py` still uses the EWMA proxy, and the `B1_REPORT.md` verdict is
final — a stronger slice baseline cannot flip an already-failed gate. AUC is
rank-based, so `dd_prob_from_logvol` (monotone in σ) leaves the EWMA arm's AUC equal
to the M2 raw-score AUC; only the **new ARIMA arm** changes any number, and only by
*raising* the baseline. The GBM AUC and the Brier skill-z deflation are untouched.


In [ ]:
# ─── T1: drawdown_21d (AUC, skill-z) ─────────────────────────────────────────
spec = TARGET_CATALOG["drawdown_21d"]
h = spec.horizon_bars
featsT, labT, pxT = make_panel(target_label_fn("drawdown_21d"))
print("T1 drawdown_21d: panel rows/symbol =", {s: len(v) for s, v in labT.items()})
gbmT = collect_oos_predictions(
    GBMModel(label_horizon=h, **GBM_PREVIEW), featsT, labT,
    train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
)
base_rate = float(np.nanmean(gbmT["y_true"].to_numpy()))

# ── T1 baseline = climatology + ARIMA-vol-implied DD probability (PRD T1 row),
#    with the M2 EWMA-vol proxy retained as a best-of floor (see markdown above;
#    PRIORITIES B1-DD-VOLIMPLIED-BASELINE). ──────────────────────────────────
# (1) ARIMA(1,0,0)-on-log-vol OOS forecast on the T1 panel (same purged WF), then
#     mapped through the parametric DD-probability function -> the PRD baseline.
vol_h = TARGET_CATALOG["realized_vol_21d"].horizon_bars
vol_label_fn = target_label_fn("realized_vol_21d")
featsV, labV = {}, {}
for s in DEMO_SYMBOLS:
    yvol = vol_label_fn(s).dropna()
    common = featsT[s].index.intersection(yvol.index)
    featsV[s], labV[s] = featsT[s].loc[common], yvol.loc[common]
arimaV = collect_oos_predictions(
    ARIMABaseline(), featsV, labV,
    train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=vol_h, embargo=EMBARGO,
)
arima_dd_prob = dd_prob_from_logvol(align_other(gbmT, arimaV))
# (2) EWMA-vol proxy mapped through the SAME function -> AUC-identical to the M2 run,
#     retained as a floor so best-of can only raise the baseline bar.
ewma_by_sym = {s: ewma_logvol(prices_by_sym[s]["close"]) for s in DEMO_SYMBOLS}
ewma_dd_prob = dd_prob_from_logvol(align_series(gbmT, ewma_by_sym))

pr_metrics, sig_ci, baseline_detail = {}, {}, {}
for rg in REGIMES:
    m = regime_mask(gbmT, rg)
    if not m.any():
        continue
    yv = gbmT["y_true"].to_numpy()[m]
    gv = gbmT["y_pred"].to_numpy()[m]
    gbm_auc = _auc(yv, gv) if len(np.unique(yv)) > 1 else np.nan
    # best-of {climatology(=0.5), ARIMA-vol-implied DD prob, EWMA-vol DD prob}
    cands = {
        "climatology": (0.5, np.full(m.sum(), base_rate)),
        "arima_vol_dd_prob": (_auc_safe(yv, arima_dd_prob[m])[0], arima_dd_prob[m]),
        "ewma_vol_dd_prob": (_auc_safe(yv, ewma_dd_prob[m])[0], ewma_dd_prob[m]),
    }
    chosen = max(cands, key=lambda k: cands[k][0] if not np.isnan(cands[k][0]) else -np.inf)
    better = cands[chosen][0]
    pr_metrics[rg] = {"auc": {"variant": gbm_auc, "baseline": float(better)}}
    base_score = cands[chosen][1]
    ok = ~np.isnan(base_score)
    variant = np.column_stack([yv[ok], gv[ok]])
    baseline = np.column_stack([yv[ok], base_score[ok]])
    try:
        sig_ci[rg] = bootstrap_metric_delta_ci(variant, baseline, _auc_cs, **BOOT)
    except ValueError:
        sig_ci[rg] = None
    baseline_detail[rg] = {"chosen": chosen} | {
        k: (round(v[0], 3) if not np.isnan(v[0]) else None) for k, v in cands.items()
    }

y_all = gbmT["y_true"].to_numpy()
brier_clim = (base_rate - y_all) ** 2
brier_gbm = (gbmT["y_pred"].to_numpy() - y_all) ** 2
skill = brier_clim - brier_gbm
skillz = forecast_skill_z(skill)
gate_results["drawdown_21d"] = b1_gate_report(
    spec, pr_metrics, sig_ci, skillz.passed,
    regimes_required=REGIMES, min_pass=MIN_PASS, deflation_detail=str(skillz),
)
score_detail["drawdown_21d"] = dict(base_rate=base_rate, skillz=skillz, baseline_detail=baseline_detail)
print(f"T1 base rate P[DD>5%] = {base_rate:.3f};  skill-z = {skillz.z:.3f} (pass={skillz.passed})")
print("T1 per-regime AUC (GBM variant / best baseline) + baseline breakdown:")
for rg, d in baseline_detail.items():
    v = pr_metrics[rg]["auc"]
    print(f"  {rg:11s} variant={v['variant']:.3f}  baseline={v['baseline']:.3f}  chosen={d['chosen']}"
          f"  (climatology={d['climatology']}, arima_vol_dd_prob={d['arima_vol_dd_prob']},"
          f" ewma_vol_dd_prob={d['ewma_vol_dd_prob']})")


In [ ]:
# ─── T2: realized_vol_21d (MAE, skill-z) ─────────────────────────────────────
spec = TARGET_CATALOG["realized_vol_21d"]
h = spec.horizon_bars
featsT, labT, pxT = make_panel(target_label_fn("realized_vol_21d"))
gbmT = collect_oos_predictions(
    GBMModel(label_horizon=h, **GBM_PREVIEW), featsT, labT,
    train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
)
arimaT = collect_oos_predictions(
    ARIMABaseline(), featsT, labT,
    train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
)
arima_pred = align_other(gbmT, arimaT)
ewma_by_sym = {s: ewma_logvol(prices_by_sym[s]["close"]) for s in DEMO_SYMBOLS}
ewma_pred = align_series(gbmT, ewma_by_sym)
y_all = gbmT["y_true"].to_numpy()
gbm_pred = gbmT["y_pred"].to_numpy()

e_ok = ~np.isnan(ewma_pred)
a_ok = ~np.isnan(arima_pred)
mae_ewma = _mae(y_all[e_ok], ewma_pred[e_ok])
mae_arima = _mae(y_all[a_ok], arima_pred[a_ok])
best_base_pred = ewma_pred if mae_ewma <= mae_arima else arima_pred
print(f"T2 aggregate MAE: EWMA={mae_ewma:.4f}  ARIMA-on-logvol={mae_arima:.4f}  GBM={_mae(y_all, gbm_pred):.4f}")

pr_metrics, sig_ci = {}, {}
for rg in REGIMES:
    m = regime_mask(gbmT, rg)
    if not m.any():
        continue
    yv = y_all[m]
    gbm_mae = _mae(yv, gbm_pred[m])
    em, am = ewma_pred[m], arima_pred[m]
    em_ok, am_ok = ~np.isnan(em), ~np.isnan(am)
    mae_e = _mae(yv[em_ok], em[em_ok]) if em_ok.any() else np.inf
    mae_a = _mae(yv[am_ok], am[am_ok]) if am_ok.any() else np.inf
    better = min(mae_e, mae_a)
    pr_metrics[rg] = {"mae": {"variant": gbm_mae, "baseline": float(better)}}
    base_pred = em if mae_e <= mae_a else am
    ok = ~np.isnan(base_pred)
    variant = np.column_stack([yv[ok], gbm_pred[m][ok]])
    baseline = np.column_stack([yv[ok], base_pred[ok]])
    try:
        sig_ci[rg] = bootstrap_metric_delta_ci(variant, baseline, _mae_cs, **BOOT)
    except ValueError:
        sig_ci[rg] = None

ok = ~np.isnan(best_base_pred)
skill = np.abs(y_all[ok] - best_base_pred[ok]) - np.abs(y_all[ok] - gbm_pred[ok])
skillz = forecast_skill_z(skill)
gate_results["realized_vol_21d"] = b1_gate_report(
    spec, pr_metrics, sig_ci, skillz.passed,
    regimes_required=REGIMES, min_pass=MIN_PASS, deflation_detail=str(skillz),
)
score_detail["realized_vol_21d"] = dict(skillz=skillz, mae_ewma=mae_ewma, mae_arima=mae_arima)
print(f"T2 skill-z (vs better baseline) = {skillz.z:.3f} (pass={skillz.passed})")
print("T2 per-regime MAE (variant / baseline):",
      {r: (round(v['mae']['variant'], 4), round(v['mae']['baseline'], 4)) for r, v in pr_metrics.items()})


In [ ]:
# ─── T3 / T4: directional (AUC + tradeable Sharpe, DSR) ──────────────────────
def score_directional(tid):
    spec = TARGET_CATALOG[tid]
    h = spec.horizon_bars
    featsT, labT, pxT = make_panel(target_label_fn(tid))
    gbmT = collect_oos_predictions(
        GBMModel(label_horizon=h, **GBM_PREVIEW), featsT, labT,
        train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
    )
    arimaT = collect_oos_predictions(
        ARIMABaseline(), featsT, labT,
        train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
    )
    arima_score = align_other(gbmT, arimaT)
    gbm_ret = simulate_signal_returns(gbmT, pxT, threshold=0.5)
    retlab = {s: fwd_return(pxT[s]["close"], h) for s in DEMO_SYMBOLS}
    feats_r, lab_r, px_r = {}, {}, {}
    for s in DEMO_SYMBOLS:
        common = featsT[s].index.intersection(retlab[s].dropna().index)
        feats_r[s] = featsT[s].loc[common]; lab_r[s] = retlab[s].loc[common]; px_r[s] = pxT[s].loc[common]
    arima_bt = run_portfolio_backtest(
        ARIMABaseline(), feats_r, lab_r, px_r,
        train_window=TRAIN_W, test_window=TEST_W, step=STEP, label_horizon=h, embargo=EMBARGO,
    )
    arima_ret = arima_bt.oos_returns

    y_all = gbmT["y_true"].to_numpy()
    gbm_pred = gbmT["y_pred"].to_numpy()
    # Tag each return series by its OWN dates (GBM and ARIMA Sharpe arms have
    # different OOS calendars from each other and from the AUC frame).
    gbm_reg = tag_regimes(gbm_ret.index, DateRangeDetector()) if len(gbm_ret) else pd.Series(dtype=object)
    ar_reg = tag_regimes(arima_ret.index, DateRangeDetector()) if len(arima_ret) else pd.Series(dtype=object)
    pr_metrics, sig_ci = {}, {}
    for rg in REGIMES:
        m = regime_mask(gbmT, rg)
        if not m.any():
            continue
        yv = y_all[m]
        gbm_auc = _auc(yv, gbm_pred[m]) if len(np.unique(yv)) > 1 else np.nan
        try:
            arima_auc = _auc(yv, arima_score[m])
        except ValueError:
            arima_auc = np.nan
        better_auc = np.nanmax([0.5, arima_auc])
        gbm_r = gbm_ret[(gbm_reg == rg).to_numpy()] if len(gbm_ret) else gbm_ret
        ar_r = arima_ret[(ar_reg == rg).to_numpy()] if len(arima_ret) else arima_ret
        gbm_sharpe = compute_metrics(gbm_r)["sharpe"] if len(gbm_r) else np.nan
        ar_sharpe = compute_metrics(ar_r)["sharpe"] if len(ar_r) else np.nan
        pr_metrics[rg] = {
            "auc": {"variant": gbm_auc, "baseline": float(better_auc)},
            "sharpe": {"variant": float(gbm_sharpe), "baseline": float(ar_sharpe)},
        }
        try:
            sig_ci[rg] = (bootstrap_sharpe_delta_ci(gbm_r, ar_r, **BOOT)
                          if (len(gbm_r) and len(ar_r)) else None)
        except ValueError:
            sig_ci[rg] = None

    try:
        dsr = deflated_sharpe_ratio(gbm_ret, LEDGER_N_SELF)
        dsr_passed = dsr.passed
    except ValueError:
        dsr, dsr_passed = None, False
    gate = b1_gate_report(
        spec, pr_metrics, sig_ci, dsr_passed,
        regimes_required=REGIMES, min_pass=MIN_PASS,
        deflation_detail=(str(dsr) if dsr else "n/a"),
    )
    return gate, dict(dsr=dsr, gbm_ret=gbm_ret, arima_ret=arima_ret)


for tid in ("directional_5d", "directional_21d"):
    print(f"Scoring {tid} ...")
    g, det = score_directional(tid)
    gate_results[tid] = g
    score_detail[tid] = det
    dsr = det["dsr"]
    msg = (f"  {tid}: GBM Sharpe={compute_metrics(det['gbm_ret'])['sharpe']:.3f}  "
           f"ARIMA Sharpe={compute_metrics(det['arima_ret'])['sharpe']:.3f}  ")
    msg += f"DSR={dsr.dsr:.3f} (pass={dsr.passed})" if dsr else "DSR n/a"
    print(msg)


## 6 · Gate verdicts (`b1_gate_report`, the source of truth)

In [ ]:
def verdict_row(tid):
    g = gate_results[tid]
    return {
        "target": tid,
        "materiality": f"{g['material_pass_count']}/{len(REGIMES)} (need {MIN_PASS})",
        "materiality_passed": g["materiality_passed"],
        "significance_passed": g["significance_passed"],
        "deflation_passed": g["deflation_passed"],
        "GATE": g["gate_passed"],
    }

verdict = pd.DataFrame([verdict_row(t) for t in TARGET_CATALOG]).set_index("target")
display(verdict)
print(f"\nAny target clears the slice gate: {bool(verdict['GATE'].any())}")
if not REAL_DATA:
    print("[REMINDER] REAL_DATA is False — this verdict is on synthetic data and is NOT valid.")


### Per-regime materiality + significance detail (every gated criterion)

In [ ]:
for tid in TARGET_CATALOG:
    g = gate_results[tid]
    print(f"\n=== {tid} ===  gate_passed={g['gate_passed']}  "
          f"deflation={g['deflation_passed']} :: {g['deflation_detail']}")
    for rg in REGIMES:
        pr = g["per_regime"].get(rg, {})
        if not pr:
            print(f"  {rg:11s} (no slice data)")
            continue
        crit = ", ".join(
            f"{c['metric']}: val={None if c['value'] is None else round(c['value'], 4)} "
            f"(thr {c['threshold']}, met={c['met']})"
            for c in pr.get("criteria", [])
        )
        ci = pr.get("significance_ci")
        ci_s = "n/a" if ci is None else f"[{ci[0]:.4f}, {ci[1]:.4f}] excl0={pr['ci_excludes_zero']}"
        print(f"  {rg:11s} mat_met={str(pr['materiality_met']):5s} {crit} | CI {ci_s}")


## 7 · Borda composite across targets (METHODOLOGY §10)

Targets carry heterogeneous metrics, so we rank them per regime by a **unitless
materiality margin** — the gated metric's delta divided by its pinned threshold
(1.0 = exactly at the bar) — then Borda-sum the ranks across the populated required
regimes. This never cherry-picks the winning regime.

In [ ]:
def materiality_margin(tid, rg):
    """Min over the target's criteria of (delta / threshold); higher = stronger edge."""
    g = gate_results[tid]
    pr = g["per_regime"].get(rg, {})
    margins = []
    for c in pr.get("criteria", []):
        v = c["value"]
        if v is None or (isinstance(v, float) and np.isnan(v)):
            margins.append(-np.inf)
        else:
            margins.append(v / c["threshold"] if c["threshold"] else v)
    return min(margins) if margins else -np.inf


def _regime_has_data(rg):
    # b1_gate_report creates a per_regime entry for EVERY required regime (even
    # empty qe_bull on this slice); a regime is "populated" only if some target
    # has a non-None criterion value there.
    for t in TARGET_CATALOG:
        for c in gate_results[t]["per_regime"].get(rg, {}).get("criteria", []):
            v = c["value"]
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                return True
    return False

populated_req = [r for r in REGIMES if _regime_has_data(r)]
margin_tbl = pd.DataFrame(
    {rg: {t: materiality_margin(t, rg) for t in TARGET_CATALOG} for rg in populated_req}
)
borda = pd.DataFrame(index=list(TARGET_CATALOG.keys()))
for rg in populated_req:
    borda[rg] = margin_tbl[rg].replace(-np.inf, np.nan).rank(method="average", na_option="bottom")
borda["borda_total"] = borda[populated_req].sum(axis=1)
borda = borda.sort_values("borda_total", ascending=False)
print("Materiality margins (gated metric delta / threshold; -inf = undefined/NaN):")
display(margin_tbl.round(3))
print("\nBorda composite (higher = stronger across regimes):")
display(borda.round(2))


## 8 · Slice verdict (provisional)

**This is a provisional slice verdict (METHODOLOGY §11).** Any target with
per-regime edge here carries to **B1-M3** for the confirmatory full-panel run; the
slice cannot ratify a B-cycle gate on its own. A *failed slice does not* by itself
trip B3/B4 — only the B1-M3 full-panel verdict does (PRD conditional skip paths).

In [ ]:
cleared = [t for t in TARGET_CATALOG if gate_results[t]["gate_passed"]]
# Carry-forward criterion = the PRD's M3 trigger ("any target showing per-regime
# edge"), i.e. materiality met in >= 1 required regime — strictly looser than the
# >= 2 strict-gate stage. The slice flags candidates; B1-M3's full panel confirms.
def regimes_with_edge(tid):
    g = gate_results[tid]
    return [r for r in REGIMES if g["per_regime"].get(r, {}).get("materiality_met")]

carry = [t for t in TARGET_CATALOG if regimes_with_edge(t)]
print("B1-M2 PROVISIONAL SLICE VERDICT")
print("=" * 64)
print(f"REAL_DATA: {REAL_DATA}   deflation N: {LEDGER_N_SELF} (ledger {LEDGER_N} + self {N_SELF_COMPARISONS})")
print(f"Targets clearing the FULL slice gate (>=2 regimes + sig + deflation): {cleared or '(none)'}")
print("Targets with per-regime EDGE (>=1 regime materiality) -> carry to B1-M3:")
for t in TARGET_CATALOG:
    rg = regimes_with_edge(t)
    print(f"    {t:18s} edge regimes: {rg or '(none)'}")
print(f"  carry-forward set: {carry or '(none)'}")
print(f"Borda slice winner: {borda.index[0]} (total {borda['borda_total'].iloc[0]:.1f})")
print("=" * 64)
print("qe_bull is empty on this 2018-start slice; the slice speaks only to covid + "
      "rate_cycle. B1-M3's full panel (2004-2026) restores all three required regimes "
      "and is the confirmatory test (METHODOLOGY §11). A failed slice does NOT by "
      "itself trip B3/B4 — only the B1-M3 full-panel verdict does.")
